In [ ]:
import requests
import os
import aiohttp
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from pydantic_settings import BaseSettings, SettingsConfigDict
ENV = Path("./.env")
#BASE_DIR = Path(__file__).resolve().parents[1] to be added

In [ ]:
import aiohttp
from pathlib import Path


async def fetch_raw_xml(
    api_key: str,
    start: str,
    end: str,
    domain: str = "10YCH-SWISSGRIDZ",
    save_dir: str = "data/raw_xml",
) -> Path:
    """
    Download raw ENTSO-E XML (Swiss total load) and save locally.

    Parameters
    ----------
    api_key : str
        ENTSO-E token
    start : str
        YYYYMMDDHHMM
    end : str
        YYYYMMDDHHMM
    domain : str
        Switzerland default
    save_dir : str
        folder to save xml files

    Returns
    -------
    pathlib.Path to saved file
    """

    url = "https://web-api.tp.entsoe.eu/api"

    params = {
        "documentType": "A65",      # total load
        "processType": "A16",       # realised
        "outBiddingZone_Domain": domain,
        "periodStart": start,
        "periodEnd": end,
        "securityToken": api_key,
    }

    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)

    file_name = f"load_{start}_{end}.xml"
    file_path = save_path / file_name

    async with aiohttp.ClientSession() as session:
        async with session.get(url, params=params) as response:
            response.raise_for_status()

            xml_text = await response.text()

            file_path.write_text(xml_text, encoding="utf-8")

    return file_path